# Data Understanding of the Heart Diagnoses file

This file contains the data understanding task applied on the Heart Diagnoses CSV file.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from src.utils import data_understanding as du

heart_diagnoses_df = pd.read_csv("../../data/raw/heart_diagnoses_1.csv")
heart_diagnoses_df['subject_id'].nunique()

In [ ]:
#print the number  of columns and rows of the dataset
heart_diagnoses_shape=heart_diagnoses_df.shape
print("Number of rows and columns in heart diagnoses dataset:",heart_diagnoses_shape)

In [ ]:
#print the columns of the dataset 
heart_diagnoses_df_columns= du.get_cols(heart_diagnoses_df)
print("Columns in heart diagnoses dataset:",heart_diagnoses_df_columns) 

In [ ]:
#print the numerical columns of the dataset
heart_diagnoses_df_num_col=du.get_numeric_cols(heart_diagnoses_df)
print("Numerical columns in heart diagnoses dataset:",heart_diagnoses_df_num_col)

In [ ]:
#print the number of duplicate rows
heart_diagnoses_df_duplicate=du.get_num_dups(heart_diagnoses_df)
print("Number of duplicate rows in heart diagnoses dataset:", heart_diagnoses_df_duplicate)

In [ ]:
#print the  missing values in each column
heart_diagnoses_df_none=du.get_cols_with_none(heart_diagnoses_df)
print("Missing values in each column of heart diagnoses dataset:\n",heart_diagnoses_df_none)

#number of missing values in each columns 
def get_num_none(df:pd.DataFrame):
  df_missing_values=df.isnull().sum()
  return df_missing_values

In [ ]:
# Count of None (NaN) values per column
print("NUMBER OF MISSING (none) VALUES PER COLUMN:\n")
none_counts = heart_diagnoses_df.isna().sum().sort_values(ascending=False)
print(none_counts)

none_counts_log = np.log10(none_counts.replace(0, np.nan))

none_df = none_counts_log.reset_index()
none_df.columns = ["column", "log10_count"]

fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(data=none_df, x="column", y="log10_count", color="#99c2ff", edgecolor="black")

plt.title("Numero di valori vuoti per colonna (scala log10)")
plt.xlabel("Colonna")
plt.ylabel("log10(Numero di valori None)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
# Add value labels on top of bars
for i, v in enumerate(none_df["log10_count"]):
    if not np.isnan(v):  # evita etichette per colonne senza None
        ax.text(i, v + max(none_df["log10_count"]) * 0.01, 
                f"{v:.2f}", ha="center", va="bottom", fontsize=9)
plt.show()

In [ ]:
#print the summary information of the dataset
print("Summary information of heart diagnoses dataset:")
heart_diagnoses_df.info()

In [ ]:
#print the statistical summary of numerical columns
print("Statistical summary of numerical columns in heart diagnoses dataset:")
du.get_basic_stats(heart_diagnoses_df)

In [ ]:
#calculate the number of hadmission for each patient
grouped_heart_diagnoses = heart_diagnoses_df.groupby('subject_id')['hadm_id'].nunique()
grouped_heart_diagnoses=grouped_heart_diagnoses.reset_index()
grouped_heart_diagnoses.columns=['subjecte_id','num_hadmissions']
print("Number of admissions for each patient in heart diagnoses dataset:\n",grouped_heart_diagnoses)
#print the max number of hadmission
max_hadmissions = grouped_heart_diagnoses['num_hadmissions'].max()
print(f"Maximum number of admissions for a single patient: {max_hadmissions}")
#print how many patient have a single hamsission
single_admission_count = (grouped_heart_diagnoses['num_hadmissions'] == 1).sum()
print(f"Number of patients with a single admission: {single_admission_count}")


In [ ]:
#count the number of rows where the values of storetime is less or equal to the values of charttime:
#to check the data quality control that the time when the diagnosis is stored (storetime) is after or equal to the time when the diagnosis is recorded(chartime)
error = heart_diagnoses_df[heart_diagnoses_df["storetime"] < heart_diagnoses_df["charttime"]]
print("rows where the values of storetime is less  to the values of charttime:")
print(error[["subject_id", "hadm_id", "storetime", "charttime"]])
count = error.shape[0]
print("Number of rows where the values of storetime is less to the values of charttime:",count)





In [ ]:
#Pint the first 10 rows of each procedure-related column. 

procedure_cols = ["CT", "MRI", "X-ray", "Ultrasound", "ECG", "CATH", "invasions"]

for col in procedure_cols:
    print(f"\nColonna: {col}")
    for i, val in enumerate(heart_diagnoses_df[col].head(10)):
        print(f"  Riga {i}: {repr(val)}")

In [ ]:
# Mean number of notes per hadmission
avg_notes_per_hadm = heart_diagnoses_df["note_seq"].mean()
print(" Avarage number of notes per hadmission:", avg_notes_per_hadm)

In [ ]:
#distribution of the number of notes per hadmission
sns.histplot(heart_diagnoses_df, x="note_seq")


In [ ]:
#Print the number of Female, Male, and the Total (F + M) and the total count  with that ICD code
# Create a table with Female, Male, and Total counts per icd_code
icd_gender_counts = heart_diagnoses_df.groupby(['icd_code', 'gender']).size().unstack(fill_value=0)
icd_gender_counts['Total'] = heart_diagnoses_df.groupby('icd_code').size()
icd_gender_counts['Missing']= heart_diagnoses_df.groupby('icd_code')['gender'].apply(lambda x: x.isna().sum())

# Rinomina le colonne
icd_gender_counts.columns = ['Female', 'Male', 'Total', 'Missing']

# Ordina per Total (decrescente)
icd_gender_counts = icd_gender_counts.sort_values('Total', ascending=False)

print("=== Gender Distribution per ICD Code ===\n")
print(icd_gender_counts)

In [ ]:
#associate at each icd_code the patology name
unique_codes = heart_diagnoses_df[["icd_code", "long_title"]].drop_duplicates()
print(unique_codes)

In [ ]:
# mean and median age for each icd_code
icd_stats = heart_diagnoses_df.groupby('icd_code')['age'].agg(['mean', 'median']).reset_index()
icd_stats.columns = ['icd_code', 'mean_age', 'median_age']
print("Mean and median age for each ICD code:\n", icd_stats)

In [ ]:
# 1. Conta il totale di pazienti per ogni ICD code
total_counts = heart_diagnoses_df.groupby('icd_code').size().reset_index(name='total')

# 2. Conta i decessi per ogni ICD code
death_counts = heart_diagnoses_df[heart_diagnoses_df['dod'].notna()].groupby('icd_code').size().reset_index(name='death_count')

# 3. Merge i due DataFrame
df_mortality = pd.merge(total_counts, death_counts, on="icd_code", how="left").fillna(0)

# 4. Calcola il mortality rate
df_mortality["death_rate"] = df_mortality["death_count"] / df_mortality["total"]

print("Mortality rate for each ICD code:\n", df_mortality)

In [ ]:
#to visualize the number of deaths for each icd_code
sns.barplot(data=df_mortality,x="icd_code",y="death_count")

In [ ]:
#to visualize the percentage of death for each icd_code
sns.barplot(data=df_mortality,x="icd_code",y="death_rate")

In [ ]:
#to visualize the number of female and men for each icd_code
sns.catplot(data=heart_diagnoses_df, x="icd_code", hue="gender",kind="count")

In [ ]:
#to visualize the distribution of the ages
sns.histplot(heart_diagnoses_df, x="age")

In [ ]:
#another visualization to understand the age of female and men in the dataset
sns.catplot(data=heart_diagnoses_df, x="gender", y="age", hue="gender", kind="box")

In [ ]:
#to visualize for each icd_code the age of the patient 
sns.boxplot(data=heart_diagnoses_df,x="icd_code",y="age")
